# Part 0: Environment Setup & Azure Infrastructure

# Microsoft Foundry Workshop — Hands-On Lab

Build AI agents from scratch using Microsoft Foundry, covering agent configuration,
tool usage, prompt engineering, multi-step reasoning, evaluation, and multi-agent orchestration.

**This notebook is self-contained** — it deploys all Azure infrastructure, configures tracing,
and walks you through progressively advanced agent patterns.

| Part | Duration | Focus |
|------|----------|-------|
| **Part 1** (Sections 0–5) | ~50 min | Guided baseline — learn all major patterns |
| **Part 2** (Sections 6–9) | ~90 min | Customize with your own data |

---
## Section 0: Environment Setup & Prerequisites

Before running any code, you need to create a Python virtual environment and install dependencies.

### Step 1: Open a Terminal in VS Code

Press `` Ctrl+` `` (or **Terminal → New Terminal**) and run the following commands:

```bash
# Install uv if you don't have it yet
# Windows (PowerShell):
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
# macOS / Linux:
# curl -LsSf https://astral.sh/uv/install.sh | sh

# Create virtual environment and install dependencies
uv venv .venv --python 3.12
uv pip install -r requirements.txt --python .venv
```

### Step 2: Select the `.venv` kernel

1. Click **"Select Kernel"** in the top-right corner of this notebook
2. Choose **"Python Environments..."**
3. Select **`.venv (Python 3.12.x)`** from this project folder

> ⚠️ **Do not run any cells** until the kernel indicator in the top-right shows `.venv`.

### Step 3: Verify setup

Once the kernel is set, run the cell below to validate everything is working.


In [ ]:
# --- Prerequisite Validation & Helper Setup ---
import subprocess, sys, platform, json, time, os, shutil
from pathlib import Path

def run_cmd(args: list[str], check: bool = True) -> subprocess.CompletedProcess:
    """Run a CLI command (OS-agnostic). Resolves .cmd/.bat on Windows automatically."""
    # On Windows, commands like 'az' are .cmd files — resolve the full path so
    # subprocess can find them without shell=True.
    exe = shutil.which(args[0])
    if exe:
        args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        print(f"Command failed: {' '.join(args)}")
        print(f"Error: {result.stderr.strip()}")
        raise RuntimeError(result.stderr.strip())
    return result

print(f"OS:      {platform.system()} {platform.release()}")
print(f"Python:  {sys.version}")
assert sys.version_info >= (3, 10), "Python 3.10+ is required"
print("  ✅ Python version OK")

az_ver = run_cmd(["az", "version"], check=False)
assert az_ver.returncode == 0, "Azure CLI not found — install from https://aka.ms/installazurecli"
print("  ✅ Azure CLI installed")

az_acct = run_cmd(["az", "account", "show"], check=False)
assert az_acct.returncode == 0, "Not logged in — run 'az login' first"
acct = json.loads(az_acct.stdout)
print(f"  ✅ Logged in as: {acct.get('user', {}).get('name', 'unknown')}")
print(f"  ✅ Subscription: {acct.get('name', 'unknown')}")

# Ensure Bicep CLI is installed (handles missing binary, broken symlinks from other OSes)
bicep_check = run_cmd(["az", "bicep", "version"], check=False)
if bicep_check.returncode != 0:
    print("  ⚙️  Bicep CLI not found — installing...")
    bicep_dir = Path.home() / ".azure" / "bin"
    bicep_dir.mkdir(parents=True, exist_ok=True)
    # Remove broken symlinks (e.g. left over from Windows/WSL/containers)
    bicep_path = bicep_dir / "bicep"
    if bicep_path.is_symlink() and not bicep_path.exists():
        bicep_path.unlink()
    run_cmd(["az", "bicep", "install"])
    print("  ✅ Bicep CLI installed")
else:
    print(f"  ✅ Bicep CLI installed ({bicep_check.stdout.strip()})")

print("\n🎉 All prerequisites met!")


### Section 0A: Deploy Azure Infrastructure

The following cells provision all required Azure resources via a **Bicep template**:

| Resource | Purpose |
|----------|---------|
| AI Services account | Foundry host (model access, agent runtime) |
| Foundry Project | Workspace for agents and traces |
| gpt-4.1 deployment | Primary LLM (80K TPM) |
| gpt-5 deployment | Latest frontier model (30K TPM) |
| gpt-5-mini deployment | Cost-efficient frontier model (30K TPM) |
| gpt-4.1-mini deployment | Fast, lightweight model (30K TPM) |
| gpt-4.1-nano deployment | Ultra-low-cost model (30K TPM) |
| Log Analytics + App Insights | Tracing backend |
| RBAC assignments | Permissions for your user |

> ⏱ Deployment takes **3–5 minutes**. You only need to run this once.


In [ ]:
# --- Configuration ---
# Generate a unique suffix from your Azure AD user ID to avoid naming collisions.

SUFFIX = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()[:6]

RESOURCE_GROUP = f"rg-foundry-workshop-{SUFFIX}"
LOCATION       = "swedencentral"   # Change if needed (e.g. eastus2, northcentralus)
FOUNDRY_NAME   = f"fndry-ws-{SUFFIX}"
PROJECT_NAME   = "workshop-project"
MODEL_NAME     = "gpt-4.1"
MODEL_VERSION  = "2025-04-14"
MODEL_CAPACITY = 80               # Thousands of tokens per minute

BICEP_PATH = Path.cwd() / "infra" / "main.bicep"

print(f"Resource Group:  {RESOURCE_GROUP}")
print(f"Location:        {LOCATION}")
print(f"Foundry Account: {FOUNDRY_NAME}")
print(f"Project:         {PROJECT_NAME}")
print(f"Bicep Path:      {BICEP_PATH}")
print(f"\nModel deployments:")
print(f"  {MODEL_NAME} v{MODEL_VERSION} — {MODEL_CAPACITY}K TPM (primary)")
print(f"  gpt-5 v2025-08-07 — 30K TPM")
print(f"  gpt-5-mini v2025-08-07 — 30K TPM")
print(f"  gpt-4.1-mini v2025-04-14 — 30K TPM")
print(f"  gpt-4.1-nano v2025-04-14 — 30K TPM")


In [ ]:
# --- Create Resource Group ---
print(f"Creating resource group '{RESOURCE_GROUP}' in {LOCATION}...")
result = run_cmd(["az", "group", "create", "-n", RESOURCE_GROUP, "-l", LOCATION, "-o", "table"])
print(result.stdout)

In [ ]:
# --- Deploy Bicep Template ---
PRINCIPAL_ID = run_cmd(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"]).stdout.strip()

print("⏳ Deploying infrastructure (3-5 min)...")
result = run_cmd([
    "az", "deployment", "group", "create",
    "-g", RESOURCE_GROUP,
    "--template-file", str(BICEP_PATH),
    "-p", f"foundryAccountName={FOUNDRY_NAME}",
    "-p", f"projectName={PROJECT_NAME}",
    "-p", f"modelName={MODEL_NAME}",
    "-p", f"modelVersion={MODEL_VERSION}",
    "-p", f"modelCapacity={MODEL_CAPACITY}",
    "-p", f"deployerPrincipalId={PRINCIPAL_ID}",
    "-p", f"location={LOCATION}",
    "-o", "table"
])
print(result.stdout)
print("✅ Infrastructure deployed!")

In [ ]:
# --- Parse Deployment Outputs ---
result = run_cmd([
    "az", "deployment", "group", "show",
    "-g", RESOURCE_GROUP, "-n", "main",
    "--query", "properties.outputs", "-o", "json"
])
outputs = json.loads(result.stdout)

PROJECT_ENDPOINT        = outputs["projectEndpoint"]["value"]
MODEL_DEPLOYMENT_NAME   = outputs["modelDeploymentName"]["value"]
APP_INSIGHTS_CONN_STR   = outputs["appInsightsConnectionString"]["value"]
ACCOUNT_NAME            = outputs["accountName"]["value"]
FOUNDRY_RESOURCE_ID     = outputs["foundryResourceId"]["value"]

print(f"Project Endpoint:   {PROJECT_ENDPOINT}")
print(f"Model Deployment:   {MODEL_DEPLOYMENT_NAME}")
print(f"App Insights:       ...{APP_INSIGHTS_CONN_STR[-30:]}")

### ✅ Infrastructure Ready!

All Azure resources are deployed. You can view your project in the
[Microsoft Foundry Portal](https://ai.azure.com).

**Deployed resources:**
- AI Services account with Foundry project
- 5 model deployments: gpt-4.1 (80K TPM), gpt-5, gpt-5-mini, gpt-4.1-mini, gpt-4.1-nano (30K TPM each)
- Application Insights for tracing
- RBAC roles assigned to your user



### Section 0B: Environment Setup

> **Tracing is automatic.** When you call `responses.create()` with an `agent_reference`,
> Foundry records server-side traces — visible in **Foundry Portal → Tracing**.
> No client-side instrumentation needed.


In [ ]:
# --- Verify Virtual Environment ---
# Dependencies were installed in Step 1. This cell confirms the kernel is using the .venv.
import sys
from pathlib import Path

venv_path = Path.cwd() / ".venv"
exe = Path(sys.executable)

# Check if the Python executable lives inside the .venv directory.
# We compare using os.path normalization to handle:
#   - macOS/Linux: symlinked venv python resolves outside .venv
#   - Windows: case-insensitive paths and backslash separators
in_venv = False
try:
    exe.relative_to(venv_path)
    in_venv = True
except ValueError:
    pass

if in_venv:
    print(f"✅ Running inside .venv: {exe}")
else:
    print(f"⚠️  Current Python: {exe}")
    print(f"    Expected .venv at: {venv_path}")
    print("    → Go back to Step 2 and select the .venv kernel before continuing!")
    raise RuntimeError("Wrong kernel — select the .venv Python interpreter")


In [ ]:
# --- Imports & Authentication ---
import os

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    FileSearchTool,
    WebSearchPreviewTool,
    FunctionTool,
)
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = AzureCliCredential()

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to: {PROJECT_ENDPOINT}")

In [ ]:
# --- Verify Connectivity ---
response = openai_client.responses.create(
    model=MODEL_DEPLOYMENT_NAME,
    input="Say 'Hello from Foundry!' in one sentence."
)
print(f"🤖 {response.output_text}")
print("\n✅ Model is responding.")


---
Setup complete. Proceed to `01-first-agent.ipynb`.